In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env in this directory

In [ ]:
client = anthropic.Anthropic(
    base_url=os.environ.get("ANTHROPIC_BASE_URL"),
    api_key=os.environ.get("ANTHROPIC_AUTH_TOKEN"),
    
)

MODEL = os.environ.get("ANTHROPIC_MODEL")

print(f"Using model: {MODEL}")
print(f"Base URL:    {os.environ.get('ANTHROPIC_BASE_URL')}")

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user",
                    "content": text,
                   }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant",
                         "content": text,
                   }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=0.0, stop_sequences=["```"]):
    param = {
        "model": MODEL,
        "max_tokens": 1024,
        "messages" : messages,
        "temperature": temperature,
    }

    if system:
        param["system"] = system
    
    message = client.messages.create(**param)
    return message.content[0].text    

In [ ]:
# Make a starting list of messages
messages = []

system_prompt = """
You are a patient math tutor.Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

# Initial message
add_user_message(
    messages, 
    "Generate a one sentence movie idea")

# Pass the list of messages
answer = chat(messages, temperature=1.0)
print(answer)

In [ ]:
# Take answer and add to assistant
add_assistant_message(messages, answer)

# Add in the user's follow-up question
add_user_message(messages, "distance is speed x time")

# Call chat again
answer = chat(messages)
print(answer)

In [ ]:
messages = []

system_prompt = """
Provide a minimalistic and concise response.
Do not offer alternatives solutions, just focus on the basic straight forward solution.
"""

# Initial message
add_user_message(
    messages, 
    "Write a Python function that checks a string for duplicate characters.",
)

# Pass the list of messages
answer = chat(messages, system=system_prompt)
print(answer)

In [ ]:
messages = []

add_user_message(
    messages,
    "Write a 1 sentence description of a fake database",
)

stream = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event)

In [ ]:
messages = []

add_user_message(
    messages,
    "Write a 1 sentence description of a fake database",
)

# this is client.message.stream different from client.message.create   stream for create
with client.messages.stream(
    model=MODEL,
    max_tokens=1024,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# the print above shows how chunk of generation are coming back. If we want to save the full response we have get_final_message
stream.get_final_message()

In [ ]:
messages = []

add_user_message(
    messages,
    "Generate a very short bridge rule as json",
)

# assistance message pre-filling -- wtf ??
add_assistant_message(
    messages,
    "```json", # but I need to know how the answer is going to be formated, right ?
)

# it turned out tha that this model does not support assistant message prefill
MODEL = os.environ.get("ANTHROPIC_MODEL", "anthropic/claude-sonnet-4-6")

MODEL = os.environ.get("ANTHROPIC_MODEL", "anthropic/claude-opus-4-6")

chat(messages)

In [ ]:
messages = []

add_user_message(
    messages,
    "Generate a very short bridge rule as json. Respond ONLY with valid JSON, no markdown fences.",
)

chat(messages)

In [ ]:
messages = []

add_user_message(
    messages,
    "Generate three different sample AWS CLI commands. Each should be very short",
)

text = chat(messages)

In [ ]:
text.strip()

In [ ]:
messages = []

add_user_message(
    messages,
    """Generate three different sample AWS CLI commands.
    Each should be very short. No explanations, just the commands.
    """
)

text = chat(messages)

In [ ]:
text